# Visual Question Answering using Bidirectional Co-Attention Fusion on Visual Genome

This VQA system is an increment to the cross-attention fusion mechanism by using Bidirectional Co-Attention, allowing questions (text) and images to attend to each other for better accuracy

* Text encoder: pre-trained base BERT (frozen)
* Vision encoder: pre-trained base ViT (frozen)
* Fusion mechanism: Bidirectional Co-Attention

Key Improvements over the previous Cross-Attention Fusion mechanism

	# Improved fusion network replacing CrossAttentionFusionNetwork.
	# Key improvements over the original:
	#   - Bidirectional co-attention (both modalities update each layer)
	#   - 6 layers instead of 2 (handles compositional multi-step reasoning)
	#   - Learned attention pooling instead of CLS-only (uses all token information)
	#   - Deeper classifier head (more capacity for high-cardinality answer prediction)
	#   - LayerNorm throughout (no BatchNorm, no batch-size constraint)

```
  # On-the-fly encoding strategy — no embedding files are written to disk or Drive
  # Problem: pre-building embeddings requires ~742 GB of storage (200 GB local disk,
  # 110 GB Drive, 742 GB RAM) because each sample stores full sequences:
  #   text: (64 tokens, 768 dims) = 196 KB + image: (197 patches, 768 dims) = 604 KB = ~800 KB/sample
  #   927K samples x 800 KB = ~742 GB — exceeds every available storage option
  # BERT and ViT run inside the training loop on each batch, embeddings exist
  # only for the duration of one forward pass (~800 MB at batch_size=8) then are
  # immediately discarded — zero disk usage, RAM stays flat throughout training
  # TRAIN_EMB / VAL_EMB / TEST_EMB constants removed — no embedding files are written
```



In [1]:
import os
import json
import math
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoFeatureExtractor
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import zipfile
import requests
from io import BytesIO
from zipfile import ZipFile
from collections import Counter
from tqdm import tqdm
import traceback
from PIL import Image
from matplotlib import pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp


#**Constants, URL and Path Definitions**


In [2]:
#DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
#DEVICE = xm.xla_device # Use XLA device for TPU (DEPRECATED)
DEVICE = torch_xla.device()
ROOT = "./visual_genome_data" # directory where the JSON dumps will be placed
IMG_DIR = "./vg_images" # directory where images downloaded for experiments are stored

# URLs from the author's homepage: https://homes.cs.washington.edu/~ranjay/visualgenome/api.html
VISUAL_GENOME_BASE = "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset"
QUESTION_ANS_ZIP = "question_answers.json.zip"
IMAGE_DATA_ZIP = "image_data.json.zip"
VG_IMAGE_PART1_URL = "https://cs.stanford.edu/people/rak248/VG_100K_2/images.zip"   # 9.2 GB (part 1 of all 108,077 images)
VG_IMAGE_PART2_URL = "https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip"  # 5.47 GB (part 2 of all 108,077 images)

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/visual_genome_backup" # Google Drive backup folder
LOCAL_BACKUP_DIR = "/content" # local folder where zip files are stored (change if needed)

# Data source mode
# 'drive' : load vg_images.zip and visual_genome_data.zip from Google Drive (DRIVE_BACKUP_DIR)
# 'local' : load vg_images.zip and visual_genome_data.zip from a local path (LOCAL_BACKUP_DIR)
# 'download' — download everything fresh from the author's URLs
DATA_SOURCE = "drive"  # set to 'drive', 'local', or 'download'

SEED = 42 #for reproducibility
TRAIN_BATCH = 32 #8
VAL_BATCH = 16
MAX_TEXT_LEN = 64  # max BERT token length — questions longer than this are truncated

# EPOCHS: raised from 4 to 10 for the deeper BidirectionalCoAttentionFusionNetwork
# The new architecture has ~50M params vs ~7.7M before — it needs more gradient updates
# to fully train all 6 co-attention layers. Lower layers stabilise first (epochs 1-3),
# upper layers only start doing useful compositional reasoning once lower layers converge.
# 4 epochs left the upper layers barely trained; 10 gives the full network time to mature.
EPOCHS = 15

# EARLY_STOPPING_PATIENCE: stop training if val_acc doesn't improve for this many epochs
# Prevents wasting compute after the model has plateaued, and avoids overfitting
# on the frequent answer classes while underfitting the rare ones.
# Set to None to disable early stopping and always run all EPOCHS.
EARLY_STOPPING_PATIENCE = 4

# Drive checkpoint directory — one .pth file saved per epoch
# Allows training to resume from the last completed epoch if the Colab
# session disconnects (Pro: ~12h limit, Pro+: ~24h limit).
# Each checkpoint stores: model weights, optimizer state, scheduler state,
# epoch number, and best_val_acc — everything needed to resume exactly.
# Set to None to disable checkpointing (e.g. for quick experiments).
CHECKPOINT_DIR = os.path.join(DRIVE_BACKUP_DIR, "checkpoints")  # set None to disable

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# Original EmbeddingDataset loaded pre-built embedding tensors from a .pt file into RAM. That required ~742 GB of storage for the full dataset (not feasible)
# Alternate appraoch would be sharding the embeddings - utilize less RAM by processing fixed batches of embeddings, but needs lots of disk storage to store the all the processed batches embeddings for later training (not feasible)
# VQADataset implements an on-the-fly encoding strategy
# - stores only raw data(image file paths and question strings)
# - Each __getitem__ loads the raw .jpg and returns it alongside the question and label.
# - BERT and ViT encoding happens inside the training loop on the GPU/TPU (so embeddings exist only for one batch duration (~800 MB at batch_size=8))
# - then that batch is immediately discarded by Python's garbage collector. Zero disk usage.
class VQADataset(Dataset):
	def __init__(self, df, local_image_dir=IMG_DIR):
		# Store the dataframe rows and image directory — no tensors loaded here
		self.df = df.reset_index(drop=True)
		self.local_image_dir = local_image_dir

	def __len__(self):
		return len(self.df)

	def __getitem__(self, idx):
		row = self.df.iloc[idx]
		img_id = int(row["image_id"])
		question = str(row["question"])
		label = int(row["label"])

		# Load the raw image from local disk — only this sample's image is in memory at a time
		img_path = os.path.join(self.local_image_dir, f"{img_id}.jpg")
		try:
			img = Image.open(img_path).convert("RGB")  # convert to RGB to handle greyscale/RGBA edge cases
		except Exception:
			# Return None for missing or corrupted images — collate_fn filters these out
			return None

		return {"image": img, "question": question, "label": label}


def vqa_collate_fn(batch):
	# Filter out None entries produced by __getitem__ when an image is missing or corrupted
	# This keeps training running smoothly even if a small number of images are unavailable
	batch = [b for b in batch if b is not None]
	if not batch or len(batch) == 1:
		return None  # entire batch was invalid — train_loop skips None batches
	return {
		"images": [b["image"]    for b in batch],  # list of PIL Images — encoded by ViT in train loop
		"questions": [b["question"] for b in batch],  # list of strings — tokenized by BERT in train loop
		"labels": torch.tensor([b["label"] for b in batch], dtype=torch.long),
	}


#**Defining Cross Attention Fusion mechanism using text and image embeddings**

In [4]:
# Cross Attention Fusion Model

class CrossAttentionBlock(nn.Module):
	# Single cross-attention block: query modality attends to key-value modality
	# Used in both directions inside each BidirectionalCoAttentionLayer
	def __init__(self, d_model, nhead=8, dim_ff=2048, dropout=0.1):
		super().__init__()
		self.mha = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=dropout)
		#self.norm1 = nn.LayerNorm(d_model)  # LayerNorm: no batch-size constraint, standard in transformers
		self.norm_q = nn.LayerNorm(d_model)  # Separate norm for query modality
		self.norm_kv = nn.LayerNorm(d_model) # Separate norm for key-value modality
		self.ff = nn.Sequential(
			nn.Linear(d_model, dim_ff),
			nn.GELU(), # GELU: smoother gradient flow than ReLU, standard in BERT/ViT
			nn.Dropout(dropout),
			nn.Linear(dim_ff, d_model)
		)
		self.norm2 = nn.LayerNorm(d_model)

	def forward(self, q, kv, key_padding_mask=None):
		# Pre-norm residual: normalise before attention (more stable than post-norm for deep networks)
		#q_norm   = self.norm1(q)
		#kv_norm  = self.norm1(kv)  # share norm weights between q and kv projections
		q_norm = self.norm_q(q)
		kv_norm = self.norm_kv(kv)  # Use separate norm for kv
		attn_out, _ = self.mha(query=q_norm, key=kv_norm, value=kv_norm, key_padding_mask=key_padding_mask)
		q = q + attn_out  # residual connection
		q = q + self.ff(self.norm2(q)) # FFN with pre-norm residual
		return q


class BidirectionalCoAttentionLayer(nn.Module):
	# One full co-attention layer: text→image cross-attention AND image→text cross-attention
	# run in parallel, then both outputs are updated simultaneously.
	# This lets each modality learn which parts of the other are relevant at each depth.
	# Contrast with original: only text attended to image, image was always the static input.
	def __init__(self, d_model, nhead=8, dim_ff=2048, dropout=0.1):
		super().__init__()
		# Text queries image: 'which image patches are relevant to each text token?'
		self.txt_to_img = CrossAttentionBlock(d_model, nhead, dim_ff, dropout)
		# Image queries text: 'which words are relevant to each image patch?' (NEW)
		self.img_to_txt = CrossAttentionBlock(d_model, nhead, dim_ff, dropout)

	def forward(self, T, I, img_pad_mask=None, txt_pad_mask=None):
		# Run both directions in parallel before updating — prevents one direction
		# from seeing the other's updated output within the same layer
		T_new = self.txt_to_img(T, I, key_padding_mask=img_pad_mask)  # text attends to image
		I_new = self.img_to_txt(I, T, key_padding_mask=txt_pad_mask)  # image attends to text
		return T_new, I_new  # both modalities carry updated, mutually-informed representations


class BidirectionalCoAttentionFusionNetwork(nn.Module):
	def __init__(self, d_img=768, d_txt=768, d=256, n_heads=8,
				 num_answers=1000, num_co_layers=4, dropout=0.3):
		super().__init__()

		# Project both modalities to a shared d-dimensional space before co-attention
		self.P_img = nn.Linear(d_img, d)
		self.P_txt = nn.Linear(d_txt, d)

		# 6 bidirectional co-attention layers — each lets both modalities update each other
		# Literature (LXMERT, ViLBERT) shows 6-12 layers needed for compositional VQA
		self.co_layers = nn.ModuleList([
			BidirectionalCoAttentionLayer(d_model=d, nhead=n_heads, dim_ff=d*4, dropout=dropout)
			for _ in range(num_co_layers)
		])

		# Learned attention pooling — a trainable query vector attends over all text output tokens and assigns a learned relevance weight to each, producing a weighted sum.
		# This replaces CLS-only pooling which discarded all non-CLS token information.
		# The pooling query is learned: it discovers which token positions matter most for answer prediction, across all training samples.
		self.pool_query = nn.Parameter(torch.zeros(1, 1, d))  # (1, 1, d) — broadcast over batch
		nn.init.normal_(self.pool_query, std=0.02)             # init like BERT token embeddings
		self.pool_attn = nn.MultiheadAttention(d, n_heads, batch_first=True, dropout=dropout)
		self.pool_norm = nn.LayerNorm(d)

		# Deeper classifier head: d→1024→512→num_answers
		# More hidden capacity than original (512→256) for high-cardinality answer prediction
		# LayerNorm replaces BatchNorm1d — no batch-size constraint, standard in transformers
		self.classifier = nn.Sequential(
			nn.Linear(d, 1024),
			nn.LayerNorm(1024),
			nn.GELU(),
			nn.Dropout(dropout),
			nn.Linear(1024, 512),
			nn.LayerNorm(512),
			nn.GELU(),
			nn.Dropout(dropout),
			nn.Linear(512, num_answers),  # final projection to answer vocabulary
		)

	def forward(self, image_embedding, text_embedding, image_mask=None, text_mask=None):
		# Project both modalities into shared d-dimensional space
		I = self.P_img(image_embedding)  # (B, Ni, d)
		T = self.P_txt(text_embedding)   # (B, Nt, d)

		# Convert padding masks: attention expects True = IGNORE (opposite of our 1=valid convention)
		img_pad_mask = (image_mask == 0) if image_mask is not None else None  # (B, Ni)
		txt_pad_mask = (text_mask  == 0) if text_mask  is not None else None  # (B, Nt)

		# Run 6 bidirectional co-attention layers
		# After each layer both T and I carry richer, mutually-informed representations
		for layer in self.co_layers:
			T, I = layer(T, I, img_pad_mask=img_pad_mask, txt_pad_mask=txt_pad_mask)

		# Learned attention pooling over text output tokens
		# pool_query (1, 1, d) attends over all Nt text tokens → weighted sum → (B, 1, d)
		# The learned query discovers which token positions matter most for answer prediction, across all training samples.
		pq = self.pool_query.expand(T.shape[0], -1, -1)  # (B, 1, d)
		pooled, _ = self.pool_attn(query=pq, key=T, value=T, key_padding_mask=txt_pad_mask)  # (B, 1, d)
		pooled = self.pool_norm(pooled.squeeze(1))             # (B, d)

		# Classify: deep head maps pooled representation to answer logits
		logits = self.classifier(pooled)  # (B, num_answers)
		return logits

#**Utilities for Data fetching and preparation**

In [5]:
import time

# prepare_vg_data: single entry point for all three data source modes
#   source='drive' — extract vg_images.zip & visual_genome_data.zip from Drive
#   source='local' — extract vg_images.zip & visual_genome_data.zip from local path
#   source='download' — download everything fresh from the author's URLs
# If 'drive' or 'local' zip files are not found, falls back to 'download' automatically
def prepare_vg_data(source=DATA_SOURCE, target_dir=ROOT, timeout=120):
    VALID_SOURCES = ('drive', 'local', 'download')
    if source not in VALID_SOURCES:
        raise ValueError(f"Invalid source '{source}'. Must be one of {VALID_SOURCES}")

    os.makedirs(target_dir, exist_ok=True) # if root dir doesn't exist, create directory
    os.makedirs(IMG_DIR, exist_ok=True)

    # extract a flat image zip into IMG_DIR
    def extract_images_zip(zip_path, label=""):
        print(f"Unzipping images{' ' + label if label else ''}: ", zip_path)
        with ZipFile(zip_path, 'r') as zf:
            members = zf.namelist()
            for member in tqdm(members, desc=f"Extracting images{' ' + label if label else ''}"):
                filename = os.path.basename(member) # flatten: ignore subdirectory structure inside zip
                if not filename:
                    continue
                out_path = os.path.join(IMG_DIR, filename)
                if not os.path.exists(out_path): # skip already extracted
                    with zf.open(member) as src, open(out_path, "wb") as dst:
                        dst.write(src.read())

    # extract visual_genome_data zip into target_dir
    def extract_data_zip(zip_path):
        print("Unzipping visual_genome_data ZIP: ", zip_path)
        with ZipFile(zip_path, 'r') as zf:
            for member in tqdm(zf.namelist(), desc="Extracting visual_genome_data"):
                zf.extract(member, target_dir)
        print("JSON files extracted to", target_dir)

    # download a file from a URL with progress bar
    def fetch(url, out_path):
        print("Downloading ZIP file from: ", url)
        fetch_request = requests.get(url, timeout=timeout, stream=True) # send a GET request to download the file in streaming mode
        if fetch_request.status_code != 200:
            raise RuntimeError(f"Failed to download ZIP file at {url} with status {fetch_request.status_code}")
        total = int(fetch_request.headers.get("content-length", 0))
        with open(out_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(out_path)) as bar:
            for chunk in fetch_request.iter_content(chunk_size=8192): # write the file in chunks (to avoid loading everything into memory)
                if chunk:
                    f.write(chunk) # only write on non-empty chunks
                    bar.update(len(chunk))
        print("Saved ZIP file in path: ", out_path)

    # run the full download flow
    def run_download():
        print("Downloading Visual Genome JSONs")
        qa_zip_path  = os.path.join(target_dir, QUESTION_ANS_ZIP) # local target to the downloaded QA pairs (zip file)
        img_zip_path = os.path.join(target_dir, IMAGE_DATA_ZIP)   # local target to the downloaded images metadata (zip file)
        img_part1_zip = os.path.join(target_dir, "images_part1.zip") # local target to image part 1 zip
        img_part2_zip = os.path.join(target_dir, "images_part2.zip") # local target to image part 2 zip

        if not os.path.exists(qa_zip_path):
            fetch(f"{VISUAL_GENOME_BASE}/{QUESTION_ANS_ZIP}", qa_zip_path)
        else:
            print("Path to the QA pairs already exists in ", qa_zip_path, "\n Using QA pairs from path")

        if not os.path.exists(img_zip_path):
            fetch(f"{VISUAL_GENOME_BASE}/{IMAGE_DATA_ZIP}", img_zip_path)
        else:
            print("Path to the images metadata already exists in ", img_zip_path, "\n Using images metadata from path")

        # Loop over both downloaded ZIP files to extract their contents
        for z in (qa_zip_path, img_zip_path):
            print("Unzipping ZIP file: ", z)
            with ZipFile(z, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
        print("Download & unzip done, JSON files saved in ", target_dir)

        # Download image ZIPs (part 1 and part 2) — together these contain all 108,077 images
        if not os.path.exists(img_part1_zip):
            fetch(VG_IMAGE_PART1_URL, img_part1_zip)
        else:
            print("Image part 1 ZIP already exists, skipping download")

        if not os.path.exists(img_part2_zip):
            fetch(VG_IMAGE_PART2_URL, img_part2_zip)
        else:
            print("Image part 2 ZIP already exists, skipping download")

        # Unzip both image ZIPs flat into IMG_DIR (flatten subdirectory structure inside zip)
        extract_images_zip(img_part1_zip, label="part 1")
        extract_images_zip(img_part2_zip, label="part 2")
        print("All images extracted to", IMG_DIR)

    if source == 'drive':
        print("Data source mode: DRIVE")
        from google.colab import drive as _drive
        _drive.mount('/content/drive')

        vg_images_zip = os.path.join(DRIVE_BACKUP_DIR, "vg_images.zip")
        vg_data_zip   = os.path.join(DRIVE_BACKUP_DIR, "visual_genome_data.zip")
        missing = []
        if not os.path.exists(vg_images_zip):
            missing.append(vg_images_zip)
        if not os.path.exists(vg_data_zip):
            missing.append(vg_data_zip)

        if missing:
            print(f"WARNING: The following ZIP files were not found in Drive ({DRIVE_BACKUP_DIR}):")
            for m in missing:
                print("  Missing:", m)
            print("Falling back to downloading from the author's URLs...")
            run_download()
        else:
            print("Found ZIP files in Drive, restoring...")
            extract_data_zip(vg_data_zip)
            extract_images_zip(vg_images_zip)
            print("Restore from Drive complete.")

    elif source == 'local':
        print("Data source mode: LOCAL")
        vg_images_zip = os.path.join(LOCAL_BACKUP_DIR, "vg_images.zip")
        vg_data_zip   = os.path.join(LOCAL_BACKUP_DIR, "visual_genome_data.zip")
        missing = []
        if not os.path.exists(vg_images_zip):
            missing.append(vg_images_zip)
        if not os.path.exists(vg_data_zip):
            missing.append(vg_data_zip)

        if missing:
            print(f"WARNING: The following ZIP files were not found locally ({LOCAL_BACKUP_DIR}):")
            for m in missing:
                print("  Missing:", m)
            print("Falling back to downloading from the author's URLs...")
            run_download()
        else:
            print("Found ZIP files locally, restoring...")
            extract_data_zip(vg_data_zip)
            extract_images_zip(vg_images_zip)
            print("Restore from local complete.")

    elif source == 'download':
        print("Data source mode: DOWNLOAD")
        run_download()

# Function to load Visual Genome QA from local JSON into a flat pandas dataframe
def load_vg_qa_local(json_path, limit=2000):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"File not found: {json_path}")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = []
    for item in data:
        image_id = item.get("id", None)
        for qa_pair in item.get("qas", []):
            records.append({
                "image_id": image_id,
                "question": str(qa_pair["question"]).strip(),
                "answer": str(qa_pair["answer"]).strip().lower()
            })
            if limit and len(records) >= limit:
                break
        if limit and len(records) >= limit:
            break

    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} QA pairs from Visual Genome")
    return df

# Load image metadata mapping
def load_vg_image_data_from_local(local_dir=ROOT):
    local_img_meta = None
    p = os.path.join(local_dir, "image_data.json")

    if os.path.exists(p):
        local_img_meta = p

    if local_img_meta is None:
        print("No image_data.json found in", local_dir, ", returing empty metadata")
        return {}

    with open(local_img_meta, "r", encoding="utf-8") as f:
        data = json.load(f)

    mapping = {}
    for rec in data:
        image_id = int(rec.get("image_id") or rec.get("id"))
        mapping[image_id] = {"url": rec.get("url"), "width": rec.get("width"), "height": rec.get("height")}

    print("Loaded image metadata for", len(mapping), "images")
    return mapping

def predownload_images_for_df(df, image_meta_map, out_dir=IMG_DIR, max_images=1000):
    os.makedirs(out_dir, exist_ok=True)
    image_ids = list(dict.fromkeys(df["image_id"].tolist()))
    cnt = 0

    for img_id in tqdm(image_ids, desc="predownloading images"):
        if cnt >= max_images:
            break

        meta = image_meta_map.get(int(img_id))

        if not meta:
            continue

        url = meta.get("url")
        if not url:
            continue

        out_path = os.path.join(out_dir, f"{int(img_id)}.jpg")
        if os.path.exists(out_path):
            cnt += 1
            continue
        '''
        try:
            r = requests.get(url, timeout=60) #60 seconds

            if r.status_code == 200:
                with open(out_path, "wb") as f:
                    f.write(r.content)
                cnt += 1
        except Exception:
            continue'''

        max_attempts = 3  # Retry up to 3 times
        for attempt in range(max_attempts):
                try:
                        r = requests.get(url, timeout=60, stream=True)  # Added stream=True
                        if r.status_code == 200:
                                with open(out_path, "wb") as f:
                                        for chunk in r.iter_content(8192):  # Write in chunks
                                                if chunk:
                                                        f.write(chunk)
                                cnt += 1
                                break  # Success, exit retry loop
                except Exception:
                        time.sleep(2)  # Wait a bit before retrying
                        if attempt == max_attempts - 1:
                                print(f"Failed to download image {img_id} after {max_attempts} attempts.")

    print("Downloaded", cnt, "images to", out_dir)

def predownload_images_for_df_concurrently(df, image_meta_map, out_dir=IMG_DIR, max_images=1000, max_workers=32, timeout=60):
    os.makedirs(out_dir, exist_ok=True)
    # unique image ids, preserve order
    image_ids = list(dict.fromkeys(df["image_id"].tolist()))
    cnt = 0
    submitted = 0
    futures = []

    def download_one(url, out_path, img_id):
        if os.path.exists(out_path):
            return True
        max_attempts = 3
        for attempt in range(1, max_attempts + 1):
            try:
                r = requests.get(url, timeout=timeout, stream=True)
                if r.status_code == 200:
                    # write in chunks to avoid mem spikes
                    with open(out_path, "wb") as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                    return True
                else:
                    # non-200, no point retrying too many times quickly
                    # small pause then retry
                    time.sleep(0.5)
            except Exception:
                # backoff a bit
                time.sleep(1.0 * attempt)
        # failed after retries
        return False

    # Submit up to max_images tasks (skip IDs with no URL/meta)
    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        for img_id in image_ids:
            if submitted >= max_images:
                break

            meta = image_meta_map.get(int(img_id))
            if not meta:
                continue

            url = meta.get("url")
            if not url:
                continue

            out_path = os.path.join(out_dir, f"{int(img_id)}.jpg")
            # if file already exists, count it and don't submit a job
            if os.path.exists(out_path):
                cnt += 1
                submitted += 1
                continue

            # submit download task
            futures.append(exe.submit(download_one, url, out_path, img_id))
            submitted += 1

        # show progress and collect results
        for f in tqdm(as_completed(futures), total=len(futures), desc="predownloading images"):
            try:
                success = f.result()
                if success:
                    cnt += 1
            except Exception as e:
                # individual download failed (already retried inside)
                # print or log optionally
                # print("Download task error:", e)
                pass

    print("Downloaded", cnt, "images to", out_dir)



#**Feature Engineering and Extraction (building and saving text embeddings and image embeddings from text and image backbones)**

In [6]:
# On-the-fly encoding — replaces build_and_save_embeddings
#
# Original build_and_save_embeddings ran BERT and ViT over the entire dataset upfront
# and saved the resulting tensors to a .pt file. For 927K+ samples the output was
# ~742 GB — too large for local disk, Drive, or RAM.
#
# encode_batch runs BERT and ViT on a single batch of raw images and question strings
# inside the training loop. The resulting embeddings are used immediately for one
# forward pass through CrossAttentionFusionNetwork, then discarded. Peak extra RAM
# is ~800 MB per batch (batch_size=8 x ~100 MB per sample) — negligible and constant.
def encode_batch(images, questions, tokenizer, img_preprocessor, text_encoder, img_encoder, device=DEVICE, max_text_len=MAX_TEXT_LEN):

	# Encode a single batch of raw PIL images and question strings into sequence embeddings.
	# Called once per batch inside train_loop and evaluate — results are used for one forward pass then garbage-collected. Nothing is written to disk.

	with torch.no_grad():  # frozen encoders (gradients not needed, saves memory and compute)
		# Tokenize questions with fixed max length so all batches produce the same (B, max_text_len) shape
		tokenized = tokenizer(
			questions, padding="max_length", truncation=True,
			max_length=max_text_len, return_tensors="pt"
		)
		tokenized = {k: v.to(device) for k, v in tokenized.items()}
		text_emb = text_encoder(**tokenized).last_hidden_state  # (B, max_text_len, D_txt)
		text_mask = tokenized["attention_mask"]                  # (B, max_text_len)

		# ViT produces 197 patch tokens: 1 CLS token + 196 spatial patches (224x224 / 16x16)
		img_inputs = img_preprocessor(images=images, return_tensors="pt")
		img_inputs = {k: v.to(device) for k, v in img_inputs.items()}
		img_emb = img_encoder(**img_inputs).last_hidden_state  # (B, 197, D_img)
		img_mask = torch.ones(img_emb.shape[:2], dtype=torch.long, device=device)  # all patches valid

		# Returns:
	  # text_emb:  (B, max_text_len, D_txt)  #BERT last hidden state
	  # img_emb:   (B, Ni, D_img)          	#ViT last hidden state (197 patch tokens)
	  # text_mask: (B, max_text_len)         #1 = real token, 0 = padding
	  # img_mask:  (B, Ni)                   #all ones (ViT has no padding)
	return text_emb, img_emb, text_mask, img_mask


#**Data Preparation (Building Dataframe for Feature Engineering and Extraction)**

In [7]:
def build_vg_dataframe(local_dir=ROOT, limit=None):
	qa_json = os.path.join(local_dir, "question_answers.json")

	if not os.path.exists(qa_json):
		raise FileNotFoundError(f"Please put 'question_answers.json' into {local_dir} or run download_and_unzip_vg_jsons().")

	print("Loading QA from", qa_json)
	df = load_vg_qa_local(qa_json, limit=limit)
	df["question"] = df["question"].astype(str)
	df["answer"] = df["answer"].astype(str).str.lower().str.strip()
	df = df[df["question"].str.len() > 0]

	# if limit:
		# df = df.sample(n=min(limit, len(df)), random_state=SEED).reset_index(drop=True)
	# else:
		#print("No limit defined, so doing a random split")
		#df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
		# raise Exception("The limit on the number of QA pairs cannot be None")

	# Using the full Visual Genome dataset (limit set to None)
	df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle

	n = len(df)
	n_train = int(0.8 * n)
	n_val = int(0.1 * n)
	train_df = df.iloc[:n_train].reset_index(drop=True)
	val_df = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
	test_df = df.iloc[n_train + n_val:].reset_index(drop=True)
	print("Split sizes -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

	return train_df, val_df, test_df

#**Evaluate and Train the Model**

In [8]:
def evaluate(model, val_dataloader, criterion, tokenizer, img_preprocessor, text_encoder, img_encoder, device=DEVICE):
    model.eval()
    # Encoders also set to eval — ensures Dropout behaves correctly even though
    # their weights are frozen; eval() affects inference-time behaviour not weight updates
    text_encoder.eval()
    img_encoder.eval()
    total_val_loss = 0.0
    predictions = []
    true_vals = []
    conf = []

    with torch.no_grad():
        for batch in val_dataloader:
            if batch is None:  # skip batches where all images were missing/corrupted
                continue

            labels = batch["labels"].to(device)

            # Encode raw images and questions on-the-fly — embeddings exist only for this batch
            text_emb, img_emb, text_mask, img_mask = encode_batch(
                batch["images"], batch["questions"],
                tokenizer, img_preprocessor, text_encoder, img_encoder, device
            )

            inputs = {
                'image_embedding': img_emb,
                'text_embedding': text_emb,
                'image_mask': img_mask,
                'text_mask': text_mask,
            }

            outputs = model(**inputs)
            loss = criterion(outputs.view(-1, outputs.shape[-1]), labels.view(-1))
            total_val_loss += loss.item()
            probs = torch.max(outputs.softmax(dim=1), dim=-1)[0].detach().cpu().numpy()
            outputs = outputs.argmax(-1)
            predictions.append(outputs.detach().cpu().numpy())
            true_vals.append(labels.cpu().numpy())
            conf.append(probs)
            torch_xla.sync() # Sync for evaluation

    loss_val_avg = total_val_loss / len(val_dataloader)
    predictions = np.concatenate(predictions, axis=0)
    true_vals = np.concatenate(true_vals, axis=0)
    conf = np.concatenate(conf, axis=0)

    return loss_val_avg, predictions, true_vals, conf


def save_checkpoint(epoch, model, optimizer, scheduler, best_val_acc, checkpoint_dir):
    # Saves a full training snapshot after every epoch to Google Drive.
    # Stores everything needed to resume exactly where training left off:
    #   - model weights (to continue training, not just inference)
    #   - optimizer state (momentum/variance accumulators for AdamW)
    #   - scheduler state (cosine cycle position)
    #   - epoch number (so resume knows which epoch to start from)
    #   - best_val_acc (so early stopping history is preserved across restarts)
    # Files are named checkpoint_epoch_N.pth, so this allows roll back to any epoch if a later one overfits
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_acc": best_val_acc,
    }, path)
    print(f"Checkpoint saved to {path}")


def load_latest_checkpoint(model, optimizer, scheduler, checkpoint_dir):
    # Scan checkpoint_dir for all checkpoint_epoch_N.pth files and load the one with the highest epoch number.
    # Returns the epoch to resume FROM (i.e. completed_epoch + 1) and the best_val_acc seen so far
    if not checkpoint_dir or not os.path.isdir(checkpoint_dir):
        print("No checkpoint directory found. (starting from epoch 1)")
        return 1, 0.0  # returns (1, 0.0) if no checkpoints exist — i.e. start fresh from epoch 1.

    # Find all checkpoint files and pick the latest by epoch number
    files = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_epoch_") and f.endswith(".pth")]
    if not files:
        print("No checkpoints found in", checkpoint_dir, "(starting from epoch 1)")
        return 1, 0.0  # returns (1, 0.0) if no checkpoints exist — i.e. start fresh from epoch 1.

    # Parse epoch numbers and find latest
    latest = max(files, key=lambda f: int(f.replace("checkpoint_epoch_", "").replace(".pth", "")))
    path   = os.path.join(checkpoint_dir, latest)
    print(f"Latest checkpoint found. Resuming from checkpoint: {path}")

    ckpt = torch.load(path, map_location="cpu")  # load to CPU first, then move to device below
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    resume_epoch = ckpt["epoch"] + 1   # next epoch to run
    best_val_acc = ckpt["best_val_acc"]
    print(f"  Resumed: completed epoch {ckpt['epoch']}, best_val_acc so far = {best_val_acc:.4f}")
    print(f"  Training will continue from epoch {resume_epoch}")
    return resume_epoch, best_val_acc


def train_loop(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler,
               tokenizer, img_preprocessor, text_encoder, img_encoder,
               epochs=EPOCHS, patience=EARLY_STOPPING_PATIENCE,
               checkpoint_dir=CHECKPOINT_DIR, device=DEVICE):

    # Resume from latest Drive checkpoint if one exists, otherwise start from epoch 1
    # This means re-running the cell after a Colab disconnect automatically picks up
    # from the last completed epoch — no manual intervention needed.
    start_epoch, best_val_acc = load_latest_checkpoint(
        model, optimizer, scheduler, checkpoint_dir
    )
    # Move model and optimizer states to the correct device after loading from CPU
    model.to(device)

    # Reconstruct early stopping counter from how many epochs have passed without
    # a best checkpoint existing at the resumed epoch — conservative: reset to 0
    # (slightly generous on patience after resume, but avoids false early-stop)
    epochs_no_improvement = 0

    if start_epoch > epochs:
        print(f"Training already completed ({epochs} epochs done). Skipping train_loop.")
        return best_val_acc

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        # Encoders stay in eval mode throughout — their weights are frozen so they never
        # need train() mode; keeping them in eval() avoids dropout affecting frozen layers
        text_encoder.eval()
        img_encoder.eval()
        total_train_loss = 0.0
        train_predictions = []
        train_true_vals = []

        for batch_idx, batch in enumerate(tqdm(train_dataloader, desc=f"Train epoch {epoch}/{epochs}")):
            if batch is None:  # skip batches where all images were missing/corrupted
                continue

            labels = batch["labels"].to(device)

            # Encode raw images and questions on-the-fly — embeddings exist only for this batch
            # then are discarded after the backward pass, keeping RAM flat across all batches
            text_emb, img_emb, text_mask, img_mask = encode_batch(
                batch["images"], batch["questions"],
                tokenizer, img_preprocessor, text_encoder, img_encoder, device
            )

            inputs = {
                'image_embedding': img_emb,
                'text_embedding': text_emb,
                'image_mask': img_mask,
                'text_mask': text_mask,
            }

            outputs = model(**inputs)
            loss = criterion(outputs.view(-1, outputs.shape[-1]), labels.view(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Clip gradients before optimizer step
            xm.optimizer_step(optimizer) # Use xm.optimizer_step for TPU

            # CosineAnnealingWarmRestarts steps per batch (not per epoch)
            # epoch-1 makes it 0-indexed; dividing by batches_per_epoch gives fractional epoch
            scheduler.step(epoch - 1 + batch_idx / len(train_dataloader))

            total_train_loss += loss.item()
            train_predictions.append(outputs.argmax(-1).detach().cpu().numpy())
            train_true_vals.append(labels.cpu().numpy())
            torch_xla.sync() # Sync after each batch

        train_preds = np.concatenate(train_predictions, axis=0)
        train_trues = np.concatenate(train_true_vals, axis=0)
        train_acc = accuracy_score(train_preds, train_trues)
        val_loss, val_preds, val_trues, _ = evaluate(
            model, val_dataloader, criterion,
            tokenizer, img_preprocessor, text_encoder, img_encoder, device
        )
        val_acc = accuracy_score(val_preds, val_trues)
        print(f"Epoch {epoch}/{epochs}: train_loss={total_train_loss/len(train_dataloader):.4f}, "
              f"train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")

        # Save per-epoch checkpoint to Drive BEFORE early stopping check
        # Saved regardless of whether val_acc improved, guarantees we can always resume from this epoch even if it wasn't the best.
        if checkpoint_dir:
            save_checkpoint(epoch, model, optimizer, scheduler, best_val_acc, checkpoint_dir)

        # ── Track best model and early stopping ──
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improvement = 0  # reset patience counter on improvement
            torch.save(model.state_dict(), "best_model_vg_fusion.pth")
            # Also save best model to Drive so it survives session disconnects
            if checkpoint_dir:
                best_path = os.path.join(checkpoint_dir, "best_model_vg_fusion.pth")
                torch.save(model.state_dict(), best_path)
            print(f"  New best val_acc: {best_val_acc:.4f} (model saved)")
        else:
            epochs_no_improvement += 1
            print(f"  No improvement for {epochs_no_improvement}/{patience} epochs"
                  if patience else "")

            # Early stopping: halt training if val_acc has not improved for `patience` epochs
            # The best checkpoint saved above is always used for testing regardless.
            if patience and epochs_no_improvement >= patience:
                print(f"Early stopping triggered at epoch {epoch}, because no val_acc improvement for {patience} consecutive epochs")
                break

    return best_val_acc

#**VQA Pipeline**

In [12]:
from transformers import AutoImageProcessor

# Replace ViTFeatureExtractor with this:
image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k", use_fast=True)
def vqa_pipeline(limit_examples=2000, top_k=1000, max_images_to_download=1000, data_source=DATA_SOURCE): #limiting to only 2000 examples for experimenting (default), None for full dataset (appox 108000 examples)
	def map_label(ans):
		return ans_to_labels.get(ans, -1)
	try:
		#--------------------------------------------------Data Preparation------------------------------------------------------------------
		# Prepare data based on source mode: 'drive', 'local', or 'download'
		# 'drive' — restores from Google Drive backup zips
		# 'local' — restores from local zip files (LOCAL_BACKUP_DIR)
		# 'download' — downloads fresh from the author's URLs
		# If drive or local zips are missing, falls back to download automatically
		prepare_vg_data(source=data_source)

		# Building dataframes (based on limit size)
		train_df, val_df, test_df = build_vg_dataframe(local_dir=ROOT, limit=limit_examples)

		# Load image meta mapping
		image_meta_map = load_vg_image_data_from_local(local_dir=ROOT)

		# Build top-K answer vocabulary (for classifying answers) — top_k=None uses ALL unique answers, top_k=N uses only the N most common
		all_answers = pd.concat([train_df["answer"], val_df["answer"], test_df["answer"]])
		ans_counts = Counter(all_answers.tolist())
		if top_k is None:
			most_common = [a for a, _ in ans_counts.most_common()]  # all unique answers, no filtering
		else:
			most_common = [a for a, _ in ans_counts.most_common(top_k)]
		ans_to_labels = {a: i for i, a in enumerate(most_common)}
		label_to_ans = {i: a for a, i in ans_to_labels.items()}
		print("Top-K answer vocab size:", len(ans_to_labels))

		train_df["label"] = train_df["answer"].apply(map_label)
		val_df["label"] = val_df["answer"].apply(map_label)
		test_df["label"] = test_df["answer"].apply(map_label)
		# drop -1s (only applies when top_k is set — when top_k=None all answers are in vocab so no rows are dropped)
		if top_k is not None:
			train_df = train_df[train_df["label"] >= 0].reset_index(drop=True)
			val_df = val_df[val_df["label"] >= 0].reset_index(drop=True)
			test_df = test_df[test_df["label"] >= 0].reset_index(drop=True)
		print("After mapping to top-K => train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

		# Pre-download images subset for speed
		print("Pre-downloading up to", max_images_to_download, "images for training/val/test")
		combined_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
		predownload_images_for_df_concurrently(combined_df, image_meta_map, out_dir=IMG_DIR, max_images=max_images_to_download)
		print("Data Preperation completed!")

    #--------------------------------------------------Feature Engineering------------------------------------------------------------------
		# Load backbones (frozen)
		tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # loads BERT's tokenizer to tokenize the question strings into many lowercased tokens and convert them into token IDs that BERT understands
		text_encoder_bert = AutoModel.from_pretrained("bert-base-uncased") # loads a pre-trained BERT model used for converting the token IDs and produces dense vector embeddings for each token
		for p in text_encoder_bert.parameters(): # leaving the parameters (tensors) of the BERT model unchanged (frozen) (to avoid re-training it when the entire fusion network learns)
			p.requires_grad = False # No need to compute gradients for each tensor during backpropagation

		#img_preprocessor = AutoFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k") # loads ViT's feature extractor that normalizes & resizes images to 224×224 patches
		img_preprocessor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
		img_encoder_vit = AutoModel.from_pretrained("google/vit-base-patch16-224-in21k") #loads the ViT model
		for p in img_encoder_vit.parameters(): # leaving the parameters (tensors) of the ViT model unchanged (frozen) (to avoid re-training it when the entire fusion network learns)
			p.requires_grad = False ## No need to compute gradients for each tensor during backpropagation

		# On-the-fly encoding — no embedding files built or saved
		# VQADataset holds only raw image paths and question strings (negligible memory)
		# encode_batch() runs BERT + ViT inside the training loop on each batch
		train_dataset = VQADataset(train_df, local_image_dir=IMG_DIR)
		val_dataset   = VQADataset(val_df,   local_image_dir=IMG_DIR)
		test_dataset  = VQADataset(test_df,  local_image_dir=IMG_DIR)
		print("Feature Engineering completed!")

    #-----------------------------------------------------Model Initialization-----------------------------------------------------------------
		# Datasets / Dataloaders
		# num_workers=0: encoding happens on the TPU inside the loop, not in DataLoader workers
		# collate_fn=vqa_collate_fn: filters out None entries from missing/corrupted images
		# drop_last=True on train: prevents a 1-sample last batch which can destabilise training
		# (e.g. 144,533 % 8 = 1). Val/test don't need it — evaluate() runs under torch.no_grad()
		train_dataloader = DataLoader(train_dataset, batch_size=TRAIN_BATCH, shuffle=True,  num_workers=0, collate_fn=vqa_collate_fn, drop_last=True)
		val_dataloader   = DataLoader(val_dataset,   batch_size=VAL_BATCH,   shuffle=False, num_workers=0, collate_fn=vqa_collate_fn)
		test_dataloader  = DataLoader(test_dataset,  batch_size=VAL_BATCH,   shuffle=False, num_workers=0, collate_fn=vqa_collate_fn)

		# Model and Training setup
		num_answers_total = len(ans_to_labels)  # always derived dynamically from vocab built above
		# BidirectionalCoAttentionFusionNetwork replaces CrossAttentionFusionNetwork
		# num_co_layers=6: 6 bidirectional layers for compositional reasoning (was 2 one-directional)
		#model = BidirectionalCoAttentionFusionNetwork(d_img=768, d_txt=768, d=512, n_heads=8, num_answers=num_answers_total, num_co_layers=6, dropout=0.3)
		model = BidirectionalCoAttentionFusionNetwork(d_img=768, d_txt=768, d=256, n_heads=8, num_answers=num_answers_total, num_co_layers=4, dropout=0.3)
		model.to(DEVICE)
		text_encoder_bert.to(DEVICE)  # move frozen encoders to TPU so encode_batch runs on accelerator
		img_encoder_vit.to(DEVICE)
		criterion = nn.CrossEntropyLoss()
		optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-5, eps=1e-8) # Renamed to optimizer

		# CosineAnnealingWarmRestarts replaces the original linear warmup+decay scheduler
		# Problem with linear schedule over 10 epochs: LR decays to ~zero by epoch 4,
		# meaning epochs 5-10 train with almost no gradient signal — wasted compute.
		# CosineAnnealingWarmRestarts periodically resets the LR on a cosine curve:
		#   T_0=2:    first restart after 2 epochs
		#   T_mult=2: each subsequent cycle doubles in length (2 → 4 → 8 epochs)
		# This lets the model escape local minima at each restart and keeps LR meaningful
		# throughout all 10 epochs. The scheduler steps per batch (not per epoch) for
		# smooth LR curves — see train_loop for the .step() call.
		scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
			optimizer, T_0=4, T_mult=2, eta_min=1e-7  # eta_min: floor LR between restarts
		)
		print("Model Initialization completed!")

    #-----------------------------------------------------Model Training-----------------------------------------------------------------
		# Train — encoders passed in so train_loop can call encode_batch() on every batch
		best_val = train_loop(
			model, train_dataloader, val_dataloader, criterion, optimizer, scheduler,
			tokenizer, img_preprocessor, text_encoder_bert, img_encoder_vit,
			epochs=EPOCHS, device=DEVICE
		)
		print("Best val acc:", best_val)

		# Test — load best model weights (prefer local, fall back to Drive copy)
		best_local = "best_model_vg_fusion.pth"
		best_drive = os.path.join(CHECKPOINT_DIR, "best_model_vg_fusion.pth") if CHECKPOINT_DIR else None
		if os.path.exists(best_local):
			#model.load_state_dict(torch.load(best_local, map_location=DEVICE))
	 		model.load_state_dict(torch.load(best_local, map_location='cpu'))
		elif best_drive and os.path.exists(best_drive):
			print("Local best model not found (loading from Drive):"), best_drive
			#model.load_state_dict(torch.load(best_drive, map_location=DEVICE))
			model.load_state_dict(torch.load(best_drive, map_location='cpu'))
		else:
			raise FileNotFoundError("No best model found locally or on Drive. Was training completed?")

		# Move model to DEVICE after loading to CPU
		model.to(DEVICE)

		test_loss, preds, truths, conf = evaluate(
			model, test_dataloader, criterion,
			tokenizer, img_preprocessor, text_encoder_bert, img_encoder_vit, device=DEVICE
		)
		test_acc, test_prec, test_recall, test_f1 = accuracy_score(preds, truths), precision_score(preds, truths, average='weighted'), \
													recall_score(preds, truths, average='weighted'), f1_score(preds, truths, average='weighted')
		print("Test loss:", test_loss)
		print("Test performance:")
		print(f"Accuracy: {test_acc:.3f}, Precision: {test_prec:.3f}, Recall: {test_recall:.3f}, F1 Score: {test_f1:.3f}")

	except Exception as e:
		print("Error in VQA pipeline due to error:", e)
		print(traceback.format_exc())

#**Execute the pipeline**

In [13]:
from google.colab import files

if __name__ == "__main__":
  '''
    # Zip and download vg_images
  !zip -r /content/vg_images.zip ./vg_images
  files.download("/content/vg_images.zip")

  # Zip and download visual_genome_data
  !zip -r /content/visual_genome_data.zip ./visual_genome_data
  files.download("/content/visual_genome_data.zip")
  '''

  # --- Set DATA_SOURCE in Cell 2 to control where data comes from ---
  # DATA_SOURCE = 'drive'    — restore from Google Drive backup (DRIVE_BACKUP_DIR)
  # DATA_SOURCE = 'local'    — restore from local zip files (LOCAL_BACKUP_DIR)
  # DATA_SOURCE = 'download' — download fresh from the author's URLs
  # If drive or local zips are missing, falls back to download automatically

  # TODO: adjust params to scale up training
  #vqa_pipeline(limit_examples=1000000, top_k=5000, max_images_to_download=1000000, data_source='download') #small params for quick experiments
  #vqa_pipeline(limit_examples=None, top_k=5000, max_images_to_download=1_000_000, data_source='local')
  #vqa_pipeline(limit_examples=None, top_k=None, max_images_to_download=1_000_000, data_source='drive') # top_k=None: use ALL unique answers, no filtering
  vqa_pipeline(limit_examples=None, top_k=10000, max_images_to_download=1_000_000, data_source=DATA_SOURCE) # using the full Visual Genome dataset (limit set to None)


Data source mode: DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found ZIP files in Drive, restoring...
Unzipping visual_genome_data ZIP:  /content/drive/MyDrive/visual_genome_backup/visual_genome_data.zip


Extracting visual_genome_data: 100%|██████████| 6/6 [00:27<00:00,  4.60s/it]


JSON files extracted to ./visual_genome_data
Unzipping images:  /content/drive/MyDrive/visual_genome_backup/vg_images.zip


Extracting images: 100%|██████████| 108249/108249 [00:00<00:00, 268208.32it/s]


Restore from Drive complete.
Loading QA from ./visual_genome_data/question_answers.json
Loaded 1445322 QA pairs from Visual Genome
Split sizes -> train: 1156257 val: 144532 test: 144533
Loaded image metadata for 108077 images
Top-K answer vocab size: 10000
After mapping to top-K => train: 927465 val: 115620 test: 116032
Pre-downloading up to 1000000 images for training/val/test


...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Feature Engineering completed!
Model Initialization completed!
Latest checkpoint found. Resuming from checkpoint: /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_15.pth
  Resumed: completed epoch 15, best_val_acc so far = 0.3956
  Training will continue from epoch 16
Training already completed (15 epochs done). Skipping train_loop.
Best val acc: 0.39559764746583637


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test loss: 2.914904105346044
Test performance:
Accuracy: 0.394, Precision: 0.570, Recall: 0.394, F1 Score: 0.452


#Train + Val Accuracies (from epoch 1 to epoch 15)

Model Initialization completed!

No checkpoints found in /content/drive/MyDrive/visual_genome_backup/checkpoints (starting from epoch 1)

Train epoch 1/15: 100%|██████████| 28983/28983 [2:03:18<00:00,  3.92it/s]
Epoch 1/15: train_loss=4.4060, train_acc=0.2571, val_loss=3.6028, val_acc=0.3088, lr=4.27e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_1.pth
  New best val_acc: 0.3088 (model saved)


Train epoch 2/15: 100%|██████████| 28983/28983 [2:01:48<00:00,  3.97it/s]
Epoch 2/15: train_loss=3.4612, train_acc=0.3249, val_loss=3.2811, val_acc=0.3418, lr=2.51e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_2.pth
  New best val_acc: 0.3418 (model saved)



Train epoch 3/15: 100%|██████████| 28983/28983 [2:01:08<00:00,  3.99it/s]
Epoch 3/15: train_loss=3.2126, train_acc=0.3507, val_loss=3.1657, val_acc=0.3553, lr=7.41e-06
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_3.pth
  New best val_acc: 0.3553 (model saved)


Train epoch 4/15: 100%|██████████| 28983/28983 [2:00:26<00:00,  4.01it/s]
Epoch 4/15: train_loss=3.1004, train_acc=0.3643, val_loss=3.1411, val_acc=0.3586, lr=1.00e-07
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_4.pth
  New best val_acc: 0.3586 (model saved)


Train epoch 5/15: 100%|██████████| 28983/28983 [2:01:22<00:00,  3.98it/s]
Epoch 5/15: train_loss=3.1814, train_acc=0.3531, val_loss=3.1133, val_acc=0.3582, lr=4.81e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_5.pth
  No improvement for 1/4 epochs


Train epoch 6/15: 100%|██████████| 28983/28983 [2:00:57<00:00,  3.99it/s]
Epoch 6/15: train_loss=3.0427, train_acc=0.3723, val_loss=3.0185, val_acc=0.3712, lr=4.27e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_6.pth
  New best val_acc: 0.3712 (model saved)


Train epoch 7/15: 100%|██████████| 28983/28983 [2:00:50<00:00,  4.00it/s]
Epoch 7/15: train_loss=2.9289, train_acc=0.3883, val_loss=2.9607, val_acc=0.3814, lr=3.46e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_7.pth
  New best val_acc: 0.3814 (model saved)


Train epoch 8/15: 100%|██████████| 28983/28983 [2:00:32<00:00,  4.01it/s]
Epoch 8/15: train_loss=2.8285, train_acc=0.4024, val_loss=2.9073, val_acc=0.3880, lr=2.51e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_8.pth
  New best val_acc: 0.3880 (model saved)


Train epoch 9/15: 100%|██████████| 28983/28983 [2:00:14<00:00,  4.02it/s]
Epoch 9/15: train_loss=2.7353, train_acc=0.4166, val_loss=2.8865, val_acc=0.3923, lr=1.55e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_9.pth
  New best val_acc: 0.3923 (model saved)


Train epoch 10/15: 100%|██████████| 28983/28983 [2:01:06<00:00,  3.99it/s]
Epoch 10/15: train_loss=2.6523, train_acc=0.4291, val_loss=2.8868, val_acc=0.3941, lr=7.41e-06
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_10.pth
  New best val_acc: 0.3941 (model saved)

Train epoch 11/15: 100%|██████████| 28983/28983 [2:01:06<00:00,  3.99it/s]
Epoch 11/20: train_loss=2.5584, train_acc=0.4404, val_loss=2.8918, val_acc=0.3955, lr=2.00e-06
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_11.pth
  New best val_acc: 0.3955 (model saved)

Train epoch 12/15: 100%|██████████| 28983/28983 [2:03:24<00:00,  3.91it/s]
Epoch 12/20: train_loss=2.5321, train_acc=0.4462, val_loss=2.8947, val_acc=0.3956, lr=1.00e-07
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_12.pth
  New best val_acc: 0.3956 (model saved)

Train epoch 13/15: 100%|██████████| 28983/28983 [2:03:02<00:00,  3.93it/s]
Epoch 13/20: train_loss=2.7606, train_acc=0.4124, val_loss=2.9197, val_acc=0.3870, lr=4.95e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_13.pth
  No improvement for 1/4 epochs

Train epoch 14/15: 100%|██████████| 28983/28983 [2:03:31<00:00,  3.91it/s]
Epoch 14/15: train_loss=2.7252, train_acc=0.4185, val_loss=2.9085, val_acc=0.3922, lr=4.81e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_14.pth
  No improvement for 2/4 epochs

Train epoch 15/15: 100%|██████████| 28983/28983 [2:01:39<00:00,  3.97it/s]
Epoch 15/15: train_loss=2.6320, train_acc=0.4311, val_loss=2.9020, val_acc=0.3940, lr=4.58e-05
Checkpoint saved to /content/drive/MyDrive/visual_genome_backup/checkpoints/checkpoint_epoch_15.pth
  No improvement for 3/4 epochs

Best val acc: 0.39559764746583637

#**Google Drive Backup (images + JSON dumps)**

In [ ]:
# Zips vg_images (part1+part2 combined) and visual_genome_data and pushes both to Google Drive

import os
import zipfile
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/visual_genome_backup" # destination folder in Drive
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# Zips all files in src_dir into zip_name and saves to dest_dir in Drive.
def zip_dir_to_drive(src_dir, zip_name, dest_dir=DRIVE_BACKUP_DIR):
    zip_path = os.path.join(dest_dir, zip_name)
    if os.path.exists(zip_path):
        print(f"{zip_name} already exists in Drive, skipping.")
        return
    all_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            all_files.append(os.path.join(root, file))
    print(f"Zipping {len(all_files)} files from {src_dir} -> {zip_path}")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file_path in tqdm(all_files, desc=f"Zipping {zip_name}"):
            arcname = os.path.relpath(file_path, src_dir) # store relative paths inside zip
            zf.write(file_path, arcname)
    print(f"Saved {zip_name} to Drive at {zip_path}")

# Zip and push all 108,077 images (combined from part1 + part2) to Drive
zip_dir_to_drive(src_dir=IMG_DIR, zip_name="vg_images.zip")

# Zip and push the JSON dumps (question_answers.json, image_data.json etc.) to Drive
zip_dir_to_drive(src_dir=ROOT, zip_name="visual_genome_data.zip")

print("Backup complete. Files saved to:", DRIVE_BACKUP_DIR)


Mounted at /content/drive
Zipping 108249 files from ./vg_images -> /content/drive/MyDrive/visual_genome_backup/vg_images.zip


Zipping vg_images.zip: 100%|██████████| 108249/108249 [07:11<00:00, 250.59it/s]


Saved vg_images.zip to Drive at /content/drive/MyDrive/visual_genome_backup/vg_images.zip
Zipping 6 files from ./visual_genome_data -> /content/drive/MyDrive/visual_genome_backup/visual_genome_data.zip


Zipping visual_genome_data.zip: 100%|██████████| 6/6 [07:23<00:00, 74.00s/it]

Saved visual_genome_data.zip to Drive at /content/drive/MyDrive/visual_genome_backup/visual_genome_data.zip
Backup complete. Files saved to: /content/drive/MyDrive/visual_genome_backup


#Future Directions:
**(for improving an VQA models accuracy on Visual Genome, (decent to best))**:

Using Bidirectional Co-Attention Fusion
* Using frozen pre-trained ViT and BERT with Bidirectional Co-Attention Fusion Network (**this model**)
* Using fine-tuned ViT and BERT with Bidirectional Co-Attention Fusion Network.
* Using frozen pre-trained CLIP with Bidirectional Co-Attention Fusion Network.
* Using fine-tuned CLIP with Bidirectional Co-Attention Fusion Network

Using Visually Grounded Bidirectional Co-Attention Fusion

* Using frozen pre-trained ViT and BERT with Bidirectional Co-Attention Fusion Network  
* Using fine-tuned ViT and BERT with Visually GroundedBidirectional Co-Attention Fusion Network.
* Using frozen pre-trained CLIP with Visually GroundedBidirectional Co-Attention Fusion Network.
* Using fine-tuned CLIP with Visually Grounded Bidirectional Co-Attention Fusion Network

Using External Knowledge (Knowledge Graph)

